### Start by reading the data and defining the project

In [1]:
import polars as pl
import plotly.express as px
from datetime import datetime

In [2]:
schema = {
    "Div": pl.Categorical,
    "Date": pl.String,  # parse manually
    "HomeTeam": pl.Categorical,
    "AwayTeam": pl.Categorical,
    "FTHG": pl.Int16,
    "FTAG": pl.Int16,
    "FTR": pl.Categorical,
    "HTHG": pl.Int16,
    "HTAG": pl.Int16,
    "HTR": pl.Categorical,
    "Season": pl.Categorical,
}

data = pl.read_csv(
    "bundesliga.csv",
    schema_overrides=schema,
    encoding="utf8",
)
data = data.with_columns(
    pl.coalesce(
        pl.col("Date").cast(pl.Utf8).str.strptime(pl.Date, "%d/%m/%y", strict=False),
        pl.col("Date").cast(pl.Utf8).str.strptime(pl.Date, "%d/%m/%Y", strict=False),
    ).alias("Date")
)
data = data.rename({c: c.strip().replace("\r", "") for c in data.columns})

data.head()

Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Season
cat,date,cat,cat,i16,i16,cat,i16,i16,cat,str
"""D1""",1993-08-07,"""Bayern Munich""","""Freiburg""",3,1,"""H""",null,null,null,"""1993-94 """
"""D1""",1993-08-07,"""Dortmund""","""Karlsruhe""",2,1,"""H""",null,null,null,"""1993-94 """
"""D1""",1993-08-07,"""Duisburg""","""Leverkusen""",2,2,"""D""",null,null,null,"""1993-94 """
"""D1""",1993-08-07,"""FC Koln""","""Kaiserslautern""",0,2,"""A""",null,null,null,"""1993-94 """
"""D1""",1993-08-07,"""Hamburg""","""Nurnberg""",5,2,"""H""",null,null,null,"""1993-94 """


### Data
Provided is a dataset of all ‘Bundesliga’ (i.e., the top German football division) matches from the 1993/1994 season to the 2017/2018 season. The dataset contains 10 features:
- Date: the date the match was played
- HomeTeam: the home team
- AwayTeam: the away team
- FTHG: the number of goals the home team scored during the whole match
- FTAG: the number of goals the away team scored during the whole match
- FTR: the full-time result, (H)ome, (A)way, or (D)raw
- HTHG: the number of goals the home team scored during the first half
- HTAG: the number of goals the away team scored during the first half
- HTR: the half time result, (H)ome, (A)way, or (D)raw

Note that HTHG, HTAG, and HTR values are missing for the 1993/1994 and 1994/1995 seasons.

### Problems
##### Question 1
We wish to know which are the best Bundesliga teams over the entire period. Define a metric, which you think is fair, to rank the teams over time and then determine the top 10 teams over the entire period.

#### Answer
To make a fair scoring metric, we want to award match points won, but also to display dominance by highlighting goal differences within matches. Furthermore, to make this more comparable across all teams that may not have been in division 1 for all these year, we want to compare it against the total games played. The last one we will consider in this example is the total number of championships won, per season. This will also award the teams that have won the league, as this is essentially the most important factor of the season.

This gives us a scoring metric of 

$$ NP = \alpha \cdot PPG + \beta \cdot GDPG + \lambda \cdot CPG $$

, where NP = Normalized Points, PPG = Points Per Game, GDPG = Goal Difference Per Game and CPG = Championships Per Game. 

In [3]:
# Award each team their points and goal differences per match and combine to one long dataframe

home = data.select([
    pl.col("Season"),
    pl.col("HomeTeam").alias("Team"),
    pl.when(pl.col("FTR") == "H").then(3)
      .when(pl.col("FTR") == "D").then(1)
      .otherwise(0).alias("Points"),
    pl.col("FTHG").alias("GF"),
    pl.col("FTAG").alias("GA")
])

away = data.select([
    pl.col("Season"),
    pl.col("AwayTeam").alias("Team"),
    pl.when(pl.col("FTR") == "A").then(3)
      .when(pl.col("FTR") == "D").then(1)
      .otherwise(0).alias("Points"),
    pl.col("FTAG").alias("GF"),
    pl.col("FTHG").alias("GA")
])

long = pl.concat([home, away])

team_season = (
    long
    .group_by(["Season", "Team"])
    .agg([
        pl.sum("Points").alias("Points"),
        pl.sum("GF").alias("GF"),
        pl.sum("GA").alias("GA")
    ])
    .with_columns(
        (pl.col("GF") - pl.col("GA")).alias("GD")
    )
)

# Champions per season (points then GD tie-break)
champions = (
    team_season
    .sort(["Season", "Points", "GD"], descending=[False, True, True])
    .group_by("Season")
    .head(1)
    .select(["Season", "Team"])
)

titles = (
    champions
    .group_by("Team")
    .agg(pl.len().alias("Championships"))
)

In [4]:
# Convert to team stats

team_stats = (
    long.group_by("Team")
    .agg([
        pl.len().alias("Games"),
        pl.sum("Points").alias("TotalPoints"),
        pl.sum("GF").alias("GF"),
        pl.sum("GA").alias("GA"),
    ])
    .join(titles, on="Team", how="left")
    .with_columns([
        (pl.col("TotalPoints") / pl.col("Games")).alias("PPG"),
        ((pl.col("GF") - pl.col("GA")) / pl.col("Games")).alias("GDPG"),
        (pl.col("Championships").fill_null(0) / pl.col("Games")).alias("CPG"),
    ])
    .with_columns(
        (
            0.6 * pl.col("PPG")
            + 0.3 * pl.col("GDPG")
            + 0.1 * pl.col("CPG")
        ).alias("NP")
    )
    .sort("NP", descending=True)
)

team_stats.head(10)

Team,Games,TotalPoints,GF,GA,Championships,PPG,GDPG,CPG,NP
cat,u32,i32,i64,i64,u32,f64,f64,f64,f64
"""Bayern Munich""",850,1820,1841,764,16,2.141176,1.267059,0.018824,1.666706
"""Dortmund""",850,1472,1505,994,5,1.731765,0.601176,0.005882,1.22
"""RB Leipzig""",68,120,123,92,null,1.764706,0.455882,0.0,1.195588
"""Leverkusen""",850,1415,1514,1065,null,1.664706,0.528235,0.0,1.157294
"""Schalke 04""",850,1333,1234,1017,null,1.568235,0.255294,0.0,1.017529
"""Werder Bremen""",850,1270,1418,1236,1,1.494118,0.214118,0.001176,0.960824
"""M'Gladbach""",34,49,65,59,null,1.441176,0.176471,0.0,0.917647
"""Stuttgart""",816,1176,1234,1160,1,1.441176,0.090686,0.001225,0.892034
"""Wolfsburg""",714,985,1065,1051,1,1.379552,0.019608,0.001401,0.833754


These are the top 10 teams in the bundesliga, given our defined metric above. Here we see Bayern Munich, Dortmund, Leverkusen and Schalke all at the top of the leader board, very much as one would assume, when asked what the best teams in the league are. Interestingly enough, we have Leipzig as number 3, which is a team that has not been in the Division 1 as long as the others. In fact they have only been in the league for the first season and then the two last seasons, as seen below. Thus, one could also consider the number of years, that the team has been in the division one as a factor, displaying its dominance in the top league, however, this would require some sort of mapping of all the team name changes over the years.

In [5]:
data.filter((pl.col("HomeTeam").is_in(["RB Leipzig", "Leipzig"])) | (pl.col("AwayTeam").is_in(["RB Leipzig", "Leipzig"]))).unique(pl.col("Season")).select("Season")

Season
str
"""2017-18 """
"""2016-17 """
"""1993-94 """


##### Question 2
We are interested in making some extra cash and want to bet on the total number of goals in football matches. We have decided to place our bets at halftime. To do this profitably, we need a model that can predict the number of goals scored in the second half. Create such a model. The choice of machine learning algorithm, evaluation metrics, train–test setup, etc., is entirely up to you.

NB: We care more about your choices, reflections etc. than the results itself.

#### Answer

In order to approach this problem, we want to consider the problem at hand. At halt time, we now the score, and our objective is to bet on what the score will be at full time, i.e. not which team will win, buy purely the total number of goals at the finish time. Therefore out data generating process will produce data that is mostly between 0-5, with occasional larger scores, but never any negative results. And also, the distribution of this data will be right skewed, thus depicting what could be a poisson distributed data generating process, and therefore we would not expect gausian deviations from the mean.

To model this we would use a Negative Binomial model, as a baseline model that would be simple, yet effective, and very interpretable. Then our next choice would be gradient boosting, to see if a more complex non-linear model would be able to capture more information out of our data. And lastly I would test a simple XGBoost model, as they generally perform very well at most problems at hand, while still being able to handle missing values, such as our two seasons of no haltime score data. The last two models are more complex models and therefore also much less interpretable, giving us the importance of our baseline model, as we only want to decrease interpretablity, if performance is proportionally better.

Firstly, to verify our assumtion of the data we want to inspect our data of full time scores. However, we should not be looking at our future data ahead of time. So we will be splitting our data into Train (1993-2012), Validation (2013-2015) and Test (2016-2018).

In [6]:
# Add full-time and half time stats
data_stat = data.with_columns(
    full_time_score=pl.col("FTHG")+pl.col("FTAG"),
    half_time_score=pl.col("HTHG")+pl.col("HTAG"),
)
data

Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Season
cat,date,cat,cat,i16,i16,cat,i16,i16,cat,str
"""D1""",1993-08-07,"""Bayern Munich""","""Freiburg""",3,1,"""H""",null,null,null,"""1993-94 """
"""D1""",1993-08-07,"""Dortmund""","""Karlsruhe""",2,1,"""H""",null,null,null,"""1993-94 """
"""D1""",1993-08-07,"""Duisburg""","""Leverkusen""",2,2,"""D""",null,null,null,"""1993-94 """
"""D1""",1993-08-07,"""FC Koln""","""Kaiserslautern""",0,2,"""A""",null,null,null,"""1993-94 """
"""D1""",1993-08-07,"""Hamburg""","""Nurnberg""",5,2,"""H""",null,null,null,"""1993-94 """
…,…,…,…,…,…,…,…,…,…,…
"""D1""",2018-05-12,"""Hoffenheim""","""Dortmund""",3,1,"""H""",1,0,"""H""","""2017-18 """
"""D1""",2018-05-12,"""Leverkusen""","""Hannover""",3,2,"""H""",2,0,"""H""","""2017-18 """
"""D1""",2018-05-12,"""Mainz""","""Werder Bremen""",1,2,"""A""",1,1,"""D""","""2017-18 """


In [7]:
# Split data
train = data_stat.filter(pl.col("Date") < datetime(2012, 6, 1))
validation = data_stat.filter((pl.col("Date") > datetime(2012, 6, 1)) & (pl.col("Date") < datetime(2015, 6, 1)))
test = data_stat.filter(pl.col("Date") > datetime(2015, 6, 1))

In [8]:
# Aggregate counts for each total-goals value
dist = (
    train
    .group_by("full_time_score")
    .agg(pl.len().alias("count"))
    .sort("full_time_score")
)

fig = px.bar(
    dist,
    x="full_time_score",
    y="count",
    title="Distribution of Full-Time Total Goals (FTHG + FTAG)",
    labels={"full_time_score": "Full-time total goals", "count": "Number of matches"},
)

fig.show()

This confirms our assumptions. Now lets model the full time scores

In [9]:
import statsmodels.api as sm
import numpy as np

In [10]:
TARGET = "full_time_score"
FEATURE = "half_time_score"

def fit_nb(train_pl, val_pl, test_pl, alpha=1.0):

    # Convert to pandas and drop missing halftime scores
    train = train_pl.drop_nulls([FEATURE]).select([FEATURE, TARGET]).to_pandas()
    val   = val_pl.drop_nulls([FEATURE]).select([FEATURE, TARGET]).to_pandas()
    test  = test_pl.drop_nulls([FEATURE]).select([FEATURE, TARGET]).to_pandas()

    y_train = train[TARGET].astype(int)
    y_val   = val[TARGET].astype(int)
    y_test  = test[TARGET].astype(int)

    X_train = sm.add_constant(train[[FEATURE]])
    X_val   = sm.add_constant(val[[FEATURE]])
    X_test  = sm.add_constant(test[[FEATURE]])

    # Fit Negative Binomial GLM
    model = sm.GLM(
        y_train,
        X_train,
        family=sm.families.NegativeBinomial(alpha=alpha)
    ).fit()

    # Predictions (expected goals)
    mu_val  = model.predict(X_val)
    mu_test = model.predict(X_test)

    return model, mu_val, mu_test, y_val, y_test


nb_model, mu_val, mu_test, y_val, y_test = fit_nb(train, validation, test)

print(nb_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:        full_time_score   No. Observations:                 5202
Model:                            GLM   Df Residuals:                     5200
Model Family:        NegativeBinomial   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -11192.
Date:                Sun, 01 Mar 2026   Deviance:                       1379.6
Time:                        01:28:44   Pearson chi2:                 1.02e+03
No. Iterations:                     6   Pseudo R-squ. (CS):             0.1069
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.5388      0.025     

We effectively have that the model says that the single input variable "half_time_score" explains roughly 10% of the variation of the target data, which in itselg might not be impressive, but given that it is a single input variable in a quick analysis, it seems fair.

The results show that the intercept has a coef of 0.53 and half_time_score has a coef of 0.35, which exponentiated gives exp(0.53) = 1.71 and exp(0.35) = 1.42, this gives us that
$$ \mathbb{E}[FT|HT=h] = 1.71 \cdot 1.42^h $$

so when there are 0 goals at half time, h = 0, the expected full-time score is expected to be 1.71. Now this seems to match with out visual observation. 

However, if the half time score is 4, then the expected full time score would be approx. 7. This indicates that it may overshoot when the half time score is high. It would be interresting to see if this performs any better than to just predict the mean, or the mode.

In [11]:
np.exp(0.5388), np.exp(0.3518) 

(np.float64(1.7139488889811156), np.float64(1.4216241704501085))

In [12]:
1.71*(1.42**4)

6.952635921599999

In [18]:
import numpy as np
from sklearn.metrics import mean_poisson_deviance

nb_val_dev = mean_poisson_deviance(
    y_val,
    np.clip(mu_val, 1e-9, None)   # avoid log(0)
)

nb_test_dev = mean_poisson_deviance(
    y_test,
    np.clip(mu_test, 1e-9, None)
)

print("NegBin — Validation Poisson deviance:", nb_val_dev)
print("NegBin — Test Poisson deviance:", nb_test_dev)

NegBin — Validation Poisson deviance: 0.6877044450490489
NegBin — Test Poisson deviance: 0.6757034310311774


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_poisson_deviance

TARGET = "full_time_score"
FEATURE = "half_time_score"

def prep_xy(df_pl):
    df = (
        df_pl
        .drop_nulls([FEATURE])
        .select([FEATURE, TARGET])
        .to_pandas()
    )
    X = df[[FEATURE]]
    y = df[TARGET].astype(int)
    return X, y

X_train, y_train = prep_xy(train)
X_val,   y_val   = prep_xy(validation)
X_test,  y_test  = prep_xy(test)

# Model
gb = HistGradientBoostingRegressor(
    loss="poisson",
    max_depth=3,
    learning_rate=0.05,
    max_iter=300,
    random_state=42,
)

gb.fit(X_train, y_train)

# Predictions (expected full-time goals)
mu_val  = gb.predict(X_val)
mu_test = gb.predict(X_test)

# Simple evaluation
print("Validation Poisson deviance:",
      mean_poisson_deviance(y_val, np.clip(mu_val, 1e-9, None)))

print("Test Poisson deviance:",
      mean_poisson_deviance(y_test, np.clip(mu_test, 1e-9, None)))

Validation Poisson deviance: 0.6877044450490489
Test Poisson deviance: 0.6757034310311774


In [ ]:
import numpy as np
from sklearn.metrics import mean_poisson_deviance
from scipy import stats

TARGET = "full_time_score"

y_train = train.drop_nulls(["half_time_score"]).select(TARGET).to_pandas()[TARGET].astype(int)
y_val   = validation.drop_nulls(["half_time_score"]).select(TARGET).to_pandas()[TARGET].astype(int)
y_test  = test.drop_nulls(["half_time_score"]).select(TARGET).to_pandas()[TARGET].astype(int)

# Constant mean predictor
mean_pred = y_train.mean()

mean_val_dev = mean_poisson_deviance(y_val, np.full(len(y_val), mean_pred))
mean_test_dev = mean_poisson_deviance(y_test, np.full(len(y_test), mean_pred))

# Constant mode predictor
mode_pred = stats.mode(y_train, keepdims=True).mode[0]

mode_val_dev = mean_poisson_deviance(y_val, np.full(len(y_val), mode_pred))
mode_test_dev = mean_poisson_deviance(y_test, np.full(len(y_test), mode_pred))

print("Constant Mean — Validation deviance:", mean_val_dev)
print("Constant Mean — Test deviance:", mean_test_dev)
print()
print("Constant Mode — Validation deviance:", mode_val_dev)
print("Constant Mode — Test deviance:", mode_test_dev)

Constant Mean — Validation deviance: 1.1538128359629667
Constant Mean — Test deviance: 1.1351863308942522

Constant Mode — Validation deviance: 1.543956799077593
Constant Mode — Test deviance: 1.4396891491011525


This essentially shows that our input feature has on monotonic interpretation, which both the NegBin model and GradBoost model are learning exactly the same. Furthermore, they are both significantly better choices than a constant mean or mode prediction. Thus they both have greater economical meaning than a constant guess. But a single monotonic input factor gives no non-linear relationship for the GradBoost to exploit, thus it is pointless to extend it to XGBoost, as this will yield the same. 

Next I would be looking at more feature engineering. One could use some Elo-ranking system, add all the features in the data set as input factors, look at the history of goals scored and conceded for each team, make some moving averages and so on.